# 11 — Estimators and the content-only baseline

**Stage question.** How are λ and rank selected, and how predictable is reply delivery from content alone?

| | |
|---|---|
| **Purpose** | Report the content-only baseline (Z_content) from out-of-fold predictions inside training, with λ choices and effective degrees of freedom. |
| **Input** | The analysis data of notebook 10. |
| **Processing** | grouped 5-fold CV over training sessions; λ per block from the relative grid. |
| **Output** | `results/baseline/` summary. |
| **Scientific assumptions** | The test split is not read here: it is touched once per condition, in J1 (notebook 16). |
| **Failure modes** | λ at the edge of the grid (flagged). |
| **What to look at** | Block out-of-fold R² and the edge flags. |

**Terms used in this notebook.**

- **Ridge** — linear regression with a penalty on coefficient size (λ).
- **Effective degrees of freedom** — how many parameters the penalised fit effectively uses.
- **Out-of-fold** — predicted by a model that did not see that session.

> **Before running.** The λ grid is set in `src/config.py`.

In [1]:
# Runtime setup — identical in every notebook.
# Colab: mounts Google Drive and uses MyDrive/coda (a copy of the repository) as
#        the project root, then installs the pinned measurement stack.
# Local: uses the repository checkout.
# Every path used below is relative to that root, so Drive and the repository
# have exactly the same layout and outputs can be copied back unchanged.
import os, sys, subprocess
from pathlib import Path

try:
    from google.colab import drive  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    ROOT = Path(os.environ.get('CODA_ROOT', '/content/drive/MyDrive/coda'))
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                    str(ROOT / 'requirements-colab.txt')], check=True)
else:
    ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

os.environ['CODA_ROOT'] = str(ROOT)
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.runtime import runtime_report
REPORT = runtime_report()

Mounted at /content/drive
runtime        colab   python 3.13.16   cpus 2
memory         12.4 GB free of 13.6 GB
gpu            none (not needed for this stage unless stated)
project disk   11.6 GB free   root = /content/drive/MyDrive/coda
scratch disk   93.4 GB free   (audio lives here briefly)
code           fingerprint 9b5146f3ab46b0fc
artifacts      manifest=True, metadata=True, transcripts=True, turns=True, audio_pass_sessions=293, channel_qc=True


**Runtime check.** The cell above prints free disk, memory, CPU, GPU, the code fingerprint and which artifacts already exist. The fingerprint identifies the exact source and configuration that ran and is written into every run record.

In [2]:
STAGE = 'sample'
STEPS = ['j1_j2']

## Settings

The method settings come from `src/settings.py` (class `Settings`).

In [3]:
from src.settings import Settings
SETTINGS = Settings()

In [4]:
from src.paths import paths
from src.experiments.stages import analysis_data
from src.experiments.layer1 import training_baseline
from src.config import RidgeConfig
P = paths()
data, *_ = analysis_data(P, STAGE, 'speaker_disjoint', SETTINGS)
base = training_baseline(data, RidgeConfig())
base

/usr/local/lib/python3.13/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (4 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/content/drive/MyDrive/coda/src/data/dataset.py:180: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False).astype(bool),


{'block_oof_r2': [0.0508035613984138,
  0.002978265357630683,
  0.1288448118234069,
  0.19768654015470855],
 'aggregate_oof_r2': 0.09507829468353998,
 'lambda_rel': [0.3, 1.0, 0.3, 0.1],
 'lambda_at_grid_edge': [False, False, False, False],
 'edf': [318.7427281220354,
  207.29457456504974,
  318.7427281220354,
  377.03402903768415],
 'n_training_pairs': 37600,
 'note': 'training out-of-fold only; the test split is not read'}

**Reading it.** Block R² from training-set out-of-fold predictions says how much of reply delivery the reply's own words, length, position, loudness offset and the replier's training identity predict. It is the reference every later Δ adds to. A λ at the grid edge would mean the grid should have extended further; it is flagged, not silently widened.